In [1]:
import numpy as np
import pandas as pd
import plotly.express as px

In [76]:
df = pd.read_csv('prediction_price.csv')
df.columns = df.columns.str.lower().str.strip().str.replace(" ", "_")
pd.set_option('display.max_columns', None)
df.head(10)

,state,city,street,zipcode,bedroom,bathroom,area,ppsq,lotarea,marketestimate,rentestimate,latitude,longitude,listedprice
0,AL,Saraland,Scott Dr,36571.0,4.0,2.0,1614.0,148.636927,0.380500,240600.0,1599.0,30.819534,-88.095960,239900.0
1,AL,Robertsdale,Cowpen Creek Rd,36567.0,3.0,2.0,1800.0,144.388889,3.200000,NaN,NaN,30.590004,-87.580376,259900.0
2,AL,Gulf Shores,Spinnaker Dr #201,36542.0,2.0,2.0,1250.0,274.000000,NaN,NaN,NaN,30.284956,-87.747920,342500.0
3,AL,Chelsea,Mallet Way,35043.0,3.0,3.0,2224.0,150.629496,0.260000,336200.0,1932.0,33.357986,-86.608700,335000.0
4,AL,Huntsville,Turtlebrook Ct,35811.0,3.0,2.0,1225.0,204.081633,NaN,222700.0,1679.0,34.775517,-86.440700,250000.0
5,AL,Montgomery,Brampton Ln,36117.0,3.0,2.0,1564.0,96.547315,0.200000,150500.0,1385.0,32.372746,-86.165115,151000.0
6,AL,Boaz,Greenwood Ave,35957.0,3.0,2.0,1717.0,139.196273,0.380000,238400.0,2125.0,34.210014,-86.136690,239000.0
7,AL,Albertville,Lexington Ave,35950.0,3.0,2.0,1674.0,149.283154,0.344353,248000.0,1597.0,34.275400,-86.217920,249900.0
8,AL,Mobile,Emerald Dr W,36619.0,3.0,3.0,2190.0,134.703196,0.344300,294000.0,1900.0,30.595074,-88.203070,295000.0
9,AL,Madison,Highpointb Plan Grayson Lndg,35756.0,4.0,3.0,3030.0,173.234323,0.340000,NaN,NaN,34.755985,-86.865920,524900.0


### Understanding Data

In [77]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22681 entries, 0 to 22680
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   state           22681 non-null  object 
 1   city            22681 non-null  object 
 2   street          22681 non-null  object 
 3   zipcode         22681 non-null  float64
 4   bedroom         22667 non-null  float64
 5   bathroom        22647 non-null  float64
 6   area            22681 non-null  float64
 7   ppsq            22681 non-null  float64
 8   lotarea         21779 non-null  float64
 9   marketestimate  15445 non-null  float64
 10  rentestimate    16705 non-null  float64
 11  latitude        22681 non-null  float64
 12  longitude       22681 non-null  float64
 13  listedprice     22681 non-null  float64
dtypes: float64(11), object(3)
memory usage: 2.4+ MB


In [78]:
df.isna().sum()

state                0
city                 0
street               0
zipcode              0
bedroom             14
bathroom            34
area                 0
ppsq                 0
lotarea            902
marketestimate    7236
rentestimate      5976
latitude             0
longitude            0
listedprice          0
dtype: int64

In [79]:
((df.isnull().sum()/len(df))*100).sort_values(ascending=False)

marketestimate    31.903355
rentestimate      26.348045
lotarea            3.976897
bathroom           0.149905
bedroom            0.061726
city               0.000000
street             0.000000
zipcode            0.000000
state              0.000000
ppsq               0.000000
area               0.000000
latitude           0.000000
longitude          0.000000
listedprice        0.000000
dtype: float64

In [80]:
df.describe()

,zipcode,bedroom,bathroom,area,ppsq,lotarea,marketestimate,rentestimate,latitude,longitude,listedprice
count,22681.000000,22667.000000,22647.000000,22681.000000,22681.000000,21779.000000,1.544500e+04,16705.000000,22681.000000,22681.000000,2.268100e+04
mean,50023.455403,3.393435,2.423299,2128.138398,222.641994,2.354870,4.870383e+05,2624.699192,39.751686,-92.299353,5.324399e+05
std,29570.312497,1.050506,1.157670,1577.512556,202.811788,16.128371,1.155986e+06,4029.614920,5.694751,16.866820,1.574922e+06
min,1002.000000,0.000000,0.000000,120.000000,1.925926,0.000000,1.570000e+04,100.000000,25.449816,-161.772780,4.888000e+03
25%,25419.000000,3.000000,2.000000,1400.000000,132.729544,0.173439,2.306000e+05,1641.000000,35.938618,-103.317760,2.250000e+05
50%,50703.000000,3.000000,2.000000,1849.000000,184.122149,0.299449,3.417000e+05,2149.000000,39.938480,-89.185210,3.449000e+05
75%,74134.000000,4.000000,3.000000,2466.000000,257.118205,0.930000,4.995000e+05,2800.000000,42.936455,-79.108376,4.999000e+05
max,99950.000000,21.000000,25.000000,99990.000000,6117.071334,800.000000,7.195920e+07,212834.000000,65.044370,-67.016030,7.600000e+07


In [81]:
df.describe(include='object')

,state,city,street
count,22681,22681,22681
unique,49,5481,19315
top,CT,Lincoln,Main St
freq,499,237,44


In [82]:
df.duplicated().sum()

np.int64(0)

### Data Cleaning

In [83]:
for col in df.select_dtypes(include=object).columns.tolist():
    print(col)
    print(df[col].nunique())
    print(df[col].unique())
    print('-'*50)

state
49
['AL' 'AK' 'AZ' 'AR' 'OK' 'MO' 'CA' 'CO' 'CT' 'OH' 'DE' 'FL' 'GA' 'ID'
 'MT' 'IL' 'IN' 'IA' 'KS' 'KY' 'LA' 'ME' 'MD' 'MA' 'MI' 'MN' 'MS' 'NE'
 'NV' 'NH' 'NJ' 'NM' 'NY' 'NC' 'ND' 'OR' 'PA' 'RI' 'SC' 'SD' 'TN' 'TX'
 'UT' 'VT' 'VA' 'WA' 'WV' 'WI' 'WY']
--------------------------------------------------
city
5481
['Saraland' 'Robertsdale' 'Gulf Shores' ... 'Granite Canon' 'Shoshoni'
 'Mills']
--------------------------------------------------
street
19315
['Scott Dr' 'Cowpen Creek Rd' 'Spinnaker Dr #201' ... 'Bar Two Dr'
 'Road 210a' 'Mason Dr']
--------------------------------------------------


In [84]:
for col in df.select_dtypes(exclude=object).columns.tolist():
    print(col)
    print(df[col].nunique())
    print(df[col].unique())
    print('-'*50)

zipcode
9264
[36571. 36567. 36542. ... 82649. 83112. 82932.]
--------------------------------------------------
bedroom
18
[ 4.  3.  2.  5.  6.  8.  1.  7.  0.  9. nan 14. 10. 13. 16. 21. 11. 12.
 18.]
--------------------------------------------------
bathroom
33
[ 2.          3.          1.          4.          5.          6.
  0.          8.          3.5         2.5         1.75        2.75
  2.25        7.          4.5         5.5        10.          9.
 11.         12.                 nan  1.5         0.75       13.
 17.         25.         14.         23.          3.0999999   2.0999999
  1.10000002  4.0999999   2.20000005  1.20000005]
--------------------------------------------------
area
4072
[ 1614.  1800.  1250. ...  9260.  4771. 10041.]
--------------------------------------------------
ppsq
20604
[148.63692689 144.38888889 274.         ... 241.80194805 196.55290102
 143.43478261]
--------------------------------------------------
lotarea
5546
[0.3805     3.2               n

In [85]:

df[df['lotarea'] == 0]

,state,city,street,zipcode,bedroom,bathroom,area,ppsq,lotarea,marketestimate,rentestimate,latitude,longitude,listedprice
74,AL,Birmingham,3rd St S,35205.0,2.0,2.0,778.0,170.951157,0.0,NaN,NaN,33.498814,-86.828170,133000.0
82,AL,Dothan,Meadowbrook Dr,36303.0,3.0,2.0,1664.0,105.168269,0.0,165000.0,1435.0,31.240427,-85.434040,175000.0
125,AL,Homewood,Sandner Ct APT D,35209.0,2.0,1.0,864.0,219.907407,0.0,NaN,NaN,33.471996,-86.786545,190000.0
170,AL,Birmingham,11th Ave S APT 202,35205.0,2.0,2.0,1566.0,214.495530,0.0,NaN,NaN,33.504580,-86.785140,335900.0
187,AL,Birmingham,Chase Ln #3015,35215.0,1.0,1.0,741.0,80.836707,0.0,58500.0,741.0,33.661690,-86.679050,59900.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
22253,WY,Cheyenne,E Prosser Rd LOT 10,82007.0,3.0,2.0,1188.0,71.548822,0.0,82800.0,356.0,41.107290,-104.796830,85000.0
22524,WY,Rozet,State Highway 51,82727.0,3.0,2.0,2100.0,66.619048,0.0,133500.0,1800.0,44.285380,-105.490370,139900.0
22553,WY,Cheyenne,Secret Valley Trl,82007.0,3.0,2.0,1280.0,67.968750,0.0,83400.0,271.0,41.076460,-104.806320,87000.0
22654,WY,Casper,E 21st St,82609.0,3.0,3.0,1800.0,202.777778,0.0,354600.0,1724.0,42.830086,-106.261370,365000.0


In [86]:
df[df['bathroom'] == 0]

,state,city,street,zipcode,bedroom,bathroom,area,ppsq,lotarea,marketestimate,rentestimate,latitude,longitude,listedprice
504,AK,Indian,Seward Hwy,99587.0,1.0,0.0,1250.0,159.920000,4.990000,NaN,4407.0,60.862892,-149.015560,199900.0
555,AK,Fairbanks,Old Wood Rd,99709.0,1.0,0.0,400.0,312.250000,6.700000,121300.0,1250.0,64.840610,-148.058850,124900.0
602,AK,Fairbanks,Pedro Dome Rd,99712.0,0.0,0.0,450.0,133.111111,1.100000,55000.0,1250.0,65.044370,-147.450780,59900.0
685,AK,Soldotna,Sandy Cir,99669.0,1.0,0.0,576.0,173.437500,2.300000,91600.0,1262.0,60.513890,-150.775970,99900.0
734,AK,Sutton,W Nancy Ln,99674.0,1.0,0.0,768.0,130.208333,1.400000,98700.0,3200.0,61.793950,-147.746100,100000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21966,WI,Hurley,Shea Rd,54534.0,2.0,0.0,358.0,362.849162,0.321258,NaN,NaN,46.257824,-90.134970,129900.0
22069,WI,Alvin,River Bank Ln,54542.0,0.0,0.0,320.0,234.375000,0.489991,NaN,NaN,46.029110,-88.837680,75000.0
22272,WY,Pinedale,Iron Shirt Ln,82941.0,1.0,0.0,450.0,322.222222,3.210000,143300.0,2550.0,43.031906,-110.088350,145000.0
22288,WY,Daniel,Beaver Ridge Rd,83115.0,1.0,0.0,350.0,400.000000,10.060000,NaN,1999.0,42.971222,-110.315216,140000.0


In [87]:
df[df['bedroom'] == 0]

,state,city,street,zipcode,bedroom,bathroom,area,ppsq,lotarea,marketestimate,rentestimate,latitude,longitude,listedprice
602,AK,Fairbanks,Pedro Dome Rd,99712.0,0.0,0.0,450.0,133.111111,1.100000,55000.0,1250.0,65.044370,-147.450780,59900.0
817,AK,Sterling,Real Property,99672.0,0.0,0.0,416.0,115.384615,0.000000,48000.0,1452.0,60.536010,-150.794330,48000.0
869,AK,Kasilof,Squirrel St,99610.0,0.0,1.0,864.0,127.893519,0.930000,109800.0,1385.0,60.260372,-151.280380,110500.0
6332,IA,Des Moines,Shaw St,50309.0,0.0,0.0,1420.0,56.338028,0.227000,77600.0,1614.0,41.581150,-93.602510,80000.0
8303,ME,East Baldwin,Bridgton Road,4024.0,0.0,2.0,3038.0,36.175115,0.250000,118100.0,1999.0,43.844776,-70.673030,109900.0
8869,MD,Baltimore,McCulloh St,21217.0,0.0,0.0,1350.0,14.814815,NaN,NaN,1700.0,39.309450,-76.635910,20000.0
8879,MD,Baltimore,Westmont Ave,21216.0,0.0,0.0,1320.0,53.030303,0.030000,67900.0,1599.0,39.300700,-76.671250,70000.0
8949,MD,North East,Red Toad Rd,21901.0,0.0,0.0,912.0,137.061404,0.309894,NaN,NaN,39.607320,-75.975630,125000.0
8956,MD,Baltimore,N Morley St,21229.0,0.0,0.0,1041.0,19.212296,0.033425,82500.0,1350.0,39.287420,-76.673100,20000.0
9395,MA,Boston,Commercial St,2109.0,0.0,1.0,500.0,300.000000,0.000000,NaN,NaN,42.367146,-71.053000,150000.0


In [88]:
# We considered zero values in bedroom,
# bathroom and area as missing values because
# houses cannot realistically have zero rooms
zero_as_missing = [ 'bathroom', 'area']
for col in zero_as_missing:
    df[col] = df[col].replace(0, np.nan)

df['bathroom'] = df['bathroom'].round(1)

for c in ['state', 'city']:
    df[c] = df[c].astype(str).str.strip().str.title()

## ppsq = listedprice/area  (data leakage)
## marketestimate (data leakage)
df.drop(columns=['marketestimate', 'street', 'zipcode'], inplace=True)
df = df.dropna(subset=['listedprice']).reset_index(drop=True)


In [89]:
low, high = df['listedprice'].quantile([0.005, 0.995])

outliers = df[(df['listedprice'] < low) | (df['listedprice'] > high)]

print("Number of outliers:", outliers.shape[0])

outliers.sort_values("listedprice")[[
    "listedprice",
    "area",
    "bedroom",
    "bathroom",
    "lotarea",
    "city",
    "state",
    "latitude",
    "ppsq",
    "longitude"
]].head(20)

Number of outliers: 228


,listedprice,area,bedroom,bathroom,lotarea,city,state,latitude,ppsq,longitude
15625,4888.0,2538.0,4.0,2.0,0.114800,Toledo,Oh,41.658035,1.925926,-83.566570
16904,4900.0,782.0,2.0,1.0,0.050000,Turtle Creek,Pa,40.412724,6.265985,-79.827255
14347,6000.0,1728.0,4.0,2.0,NaN,Syracuse,Ny,43.014565,3.472222,-76.142624
6665,7500.0,3000.0,2.0,1.0,0.054522,Ackley,Ia,42.554363,2.500000,-93.053400
6992,8000.0,1000.0,2.0,1.0,NaN,Wichita,Ks,37.652090,8.000000,-97.398400
18665,10000.0,2080.0,3.0,3.0,4.880000,Sparta,Tn,35.921963,4.807692,-85.493490
12150,10000.0,1972.0,4.0,2.0,NaN,Glenvil,Ne,40.503395,5.070994,-98.255530
21460,12500.0,1100.0,2.0,1.0,0.280000,Princeton,Wv,37.383610,11.363636,-81.094930
21458,12500.0,1610.0,4.0,2.0,1.460000,Harts,Wv,38.000200,7.763975,-82.098274
15995,12900.0,1194.0,2.0,1.0,0.161000,Ponca City,Ok,36.709100,10.804020,-97.082140


In [90]:
outliers.sort_values(
    "listedprice",
    ascending=False
)[[
    "listedprice",
    "area",
    "bedroom",
    "bathroom",
    "lotarea",
    "city",
    "ppsq",
    "state"
]].head(20)

,listedprice,area,bedroom,bathroom,lotarea,city,ppsq,state
12482,76000000.0,14197.0,7.0,NaN,4.972000,Incline Village,5353.243643,Nv
11568,72000000.0,45000.0,2.0,2.0,348.000000,Dayton,1600.000000,Mt
19451,65000000.0,22000.0,8.0,12.0,8.978000,Houston,2954.545455,Tx
18777,65000000.0,10626.0,5.0,9.0,383.620000,Franklin,6117.071334,Tn
12553,49500000.0,16232.0,8.0,NaN,5.140000,Crystal Bay,3049.531789,Nv
18755,40000000.0,19811.0,5.0,10.0,49.720000,Nashville,2019.080309,Tn
22462,37500000.0,9696.0,4.0,7.0,4.530000,Wilson,3867.574257,Wy
8679,34900000.0,30000.0,5.0,10.0,9.650000,Potomac,1163.333333,Md
22312,32750000.0,7984.0,5.0,7.0,37.060000,Jackson,4101.953908,Wy
9502,31000000.0,10858.0,5.0,8.0,0.099862,Boston,2855.037760,Ma


In [91]:
df.sort_values("ppsq").head(20)

,state,city,bedroom,bathroom,area,ppsq,lotarea,rentestimate,latitude,longitude,listedprice
15625,Oh,Toledo,4.0,2.0,2538.0,1.925926,0.114800,964.0,41.658035,-83.566570,4888.0
6665,Ia,Ackley,2.0,1.0,3000.0,2.500000,0.054522,NaN,42.554363,-93.053400,7500.0
14347,Ny,Syracuse,4.0,2.0,1728.0,3.472222,NaN,2250.0,43.014565,-76.142624,6000.0
18665,Tn,Sparta,3.0,3.0,2080.0,4.807692,4.880000,1999.0,35.921963,-85.493490,10000.0
12150,Ne,Glenvil,4.0,2.0,1972.0,5.070994,NaN,1300.0,40.503395,-98.255530,10000.0
15611,Oh,Steubenville,5.0,2.0,3000.0,5.333333,0.500000,NaN,40.358940,-80.625060,16000.0
14928,Nd,Hunter,7.0,11.0,25496.0,5.883276,1.580000,2674.0,47.191130,-97.213760,150000.0
16904,Pa,Turtle Creek,2.0,1.0,782.0,6.265985,0.050000,963.0,40.412724,-79.827255,4900.0
14376,Ny,Syracuse,4.0,2.0,2016.0,6.448413,NaN,2258.0,43.025500,-76.143570,13000.0
6109,In,Gary,6.0,3.0,3225.0,6.821705,0.086000,1994.0,41.601406,-87.334950,22000.0


In [92]:
df.sort_values("ppsq", ascending=False).head(20)

,state,city,bedroom,bathroom,area,ppsq,lotarea,rentestimate,latitude,longitude,listedprice
18777,Tn,Franklin,5.0,9.0,10626.0,6117.071334,383.620000,32029.0,35.927906,-87.083510,65000000.0
2365,Ca,Aptos,3.0,2.0,1859.0,5379.236148,0.183999,9602.0,36.957348,-121.888050,10000000.0
12482,Nv,Incline Village,7.0,NaN,14197.0,5353.243643,4.972000,212834.0,39.235850,-119.940020,76000000.0
2026,Ca,Encinitas,4.0,3.0,1757.0,4439.385316,0.107323,15532.0,33.046840,-117.297240,7800000.0
11699,Mt,Virginia City,4.0,3.0,2280.0,4166.666667,379.250000,29736.0,45.245230,-111.887850,9500000.0
22312,Wy,Jackson,5.0,7.0,7984.0,4101.953908,37.060000,92603.0,43.589115,-110.776680,32750000.0
22462,Wy,Wilson,4.0,7.0,9696.0,3867.574257,4.530000,124969.0,43.578857,-110.803955,37500000.0
17508,Ri,Westerly,6.0,6.0,4976.0,3616.358521,0.980005,NaN,41.314175,-71.841860,17995000.0
1698,Ar,Huntington,3.0,2.0,1400.0,3428.571429,70.010000,NaN,35.136562,-94.289154,4800000.0
22383,Wy,Jackson,7.0,9.0,8237.0,3277.892437,4.640000,NaN,43.500060,-110.769295,27000000.0


In [93]:
# remove unrealistic low values only
df = df[df["ppsq"] >= 5]

# remove clearly invalid cheap houses
df = df[df["listedprice"] >= 20000]

df = df.drop(columns=["ppsq"])

In [94]:
from datasist.structdata import detect_outliers
numeric_lst = df.select_dtypes(include = 'number').columns
for col in numeric_lst :
    outliers = detect_outliers(data = df , n = 0 , features = [col])
    print(col)
    print(outliers)
    print(len(outliers)/df[col].shape[0]*100)
    print('-'*50)

bedroom
[]
0.0
--------------------------------------------------
bathroom
[]
0.0
--------------------------------------------------
area
[27, 139, 145, 169, 278, 347, 444, 451, 513, 519, 556, 578, 582, 621, 649, 651, 680, 719, 743, 745, 747, 771, 800, 816, 835, 840, 855, 861, 905, 906, 924, 928, 968, 973, 1005, 1032, 1076, 1109, 1117, 1158, 1225, 1234, 1264, 1280, 1281, 1308, 1311, 1336, 1358, 1403, 1449, 1497, 1504, 1534, 1657, 1694, 1768, 1775, 1788, 1837, 1840, 1874, 1877, 1933, 1972, 1995, 2054, 2056, 2120, 2199, 2208, 2217, 2245, 2268, 2279, 2289, 2302, 2308, 2310, 2314, 2315, 2320, 2328, 2343, 2354, 2355, 2366, 2378, 2405, 2412, 2426, 2433, 2451, 2453, 2455, 2460, 2472, 2474, 2476, 2484, 2496, 2501, 2503, 2506, 2511, 2526, 2535, 2541, 2545, 2546, 2560, 2564, 2569, 2575, 2598, 2602, 2605, 2606, 2618, 2630, 2636, 2638, 2645, 2646, 2652, 2656, 2678, 2688, 2705, 2715, 2720, 2723, 2729, 2731, 2740, 2741, 2743, 2744, 2746, 2753, 2754, 2763, 2775, 2783, 2785, 2795, 2803, 2805, 2808, 28

### Feature Engineering

In [117]:
df['area'] = df['area'].replace(0, np.nan)
df['lotarea'] = df['lotarea'].replace(0, np.nan)
df['lot_to_area_ratio'] = df['lotarea'] / df['area']

In [118]:
df.shape

(22658, 11)

In [119]:
((df.isnull().sum()/len(df))*100).sort_values(ascending=False)

rentestimate         26.343896
lotarea               4.916586
lot_to_area_ratio     4.916586
bathroom              0.485480
bedroom               0.061788
city                  0.000000
state                 0.000000
area                  0.000000
latitude              0.000000
longitude             0.000000
listedprice           0.000000
dtype: float64

### Analysis

## Preprocessing Pipelines

In [120]:
num_cols = ['bedroom','bathroom','area','lotarea','rentestimate',
            'latitude','longitude','lot_to_area_ratio',]
ohe_cat  = ['state']
be_cat = ['city']


In [121]:
x = df.drop('listedprice', axis=1)
y = df['listedprice']


## PipeLine

### Numerical Pipeline

In [122]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler

simple_imputer = SimpleImputer(strategy='median')
rc = RobustScaler()

num_pipe = Pipeline([('SimpleImputer', simple_imputer), ('Scaling', rc)])

## Categorical Pipelines

In [123]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import  OneHotEncoder


ohe_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(drop='first', handle_unknown='ignore'))
])
ohe_pipe

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('imputer', ...), ('ohe', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'most_frequent'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. If

In [124]:
from category_encoders import BinaryEncoder

be = BinaryEncoder()

be_pipe = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('BE', be)
])
be_pipe

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('imputer', ...), ('BE', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'most_frequent'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. If 

In [125]:
from sklearn.compose import ColumnTransformer 

preprocessing= ColumnTransformer([
    ('num', num_pipe, num_cols),
    ('ohe', ohe_pipe, ohe_cat),
    ('binary', be_pipe, be_cat)
])
display(preprocessing)


,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('ohe', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e

In [126]:
'''from sklearn.model_selection import train_test_split

x = df.drop(columns=['listedprice'])
y = df['listedprice']

x_train,x_test,y_train,y_test = train_test_split(
    x,y, test_size=0.2, random_state=42
)
print('Train:', x_train.shape, ' Test:',x_test.shape) '''

"from sklearn.model_selection import train_test_split\n\nx = df.drop(columns=['listedprice'])\ny = df['listedprice']\n\nx_train,x_test,y_train,y_test = train_test_split(\n    x,y, test_size=0.2, random_state=42\n)\nprint('Train:', x_train.shape, ' Test:',x_test.shape) "

In [127]:
print(df.columns.tolist())

['state', 'city', 'bedroom', 'bathroom', 'area', 'lotarea', 'rentestimate', 'latitude', 'longitude', 'listedprice', 'lot_to_area_ratio']


### Algorithm Selection

In [128]:
((df.isnull().sum()/len(df))*100).sort_values(ascending=False)

rentestimate         26.343896
lotarea               4.916586
lot_to_area_ratio     4.916586
bathroom              0.485480
bedroom               0.061788
city                  0.000000
state                 0.000000
area                  0.000000
latitude              0.000000
longitude             0.000000
listedprice           0.000000
dtype: float64

In [129]:
from sklearn.pipeline import Pipeline  
from sklearn.model_selection import cross_validate, KFold
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.compose import TransformedTargetRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import cross_validate


cv = KFold(n_splits=5, shuffle=True, random_state=42)
models = [
    ('Linear Regression', LinearRegression()),
    ('Ridge', Ridge(random_state=42)),
    ('Lasso', Lasso(random_state=42)),
    ('KNN', KNeighborsRegressor()),
    ('Decision Tree', DecisionTreeRegressor(random_state=42)),
    ('Random Forest', RandomForestRegressor(random_state=42, n_jobs=-1)),
    ('Gradient Boosting', GradientBoostingRegressor(random_state=42)),
    ('XGBoost', XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)),]

for name, model in models:

    pipeline = Pipeline([
        ('preprocessing', preprocessing),
        ('model', model)
    ])

    model_final = TransformedTargetRegressor(
        regressor=pipeline,
        func=np.log1p,
        inverse_func=np.expm1
    )

    result = cross_validate(
        model_final,x,y,cv=cv,scoring='r2',return_train_score=True,n_jobs=-1
    )

    print(name)
    print('train score:', result['train_score'].mean())
    print('val score:', result['test_score'].mean())
    print('-' * 40)

Linear Regression
train score: -1491813.8728101705
val score: -571150181.0167472
----------------------------------------
Ridge
train score: -1487365.800525675
val score: -568491556.57285
----------------------------------------
Lasso
train score: -0.26246822394283065
val score: -0.8697517616787493
----------------------------------------
KNN
train score: 0.6461553634519109
val score: 0.5794466007807049
----------------------------------------
Decision Tree
train score: 1.0
val score: 0.18077911859192317
----------------------------------------
Random Forest
train score: 0.7949367616651063
val score: 0.5093688257781517
----------------------------------------
Gradient Boosting
train score: 0.6286475467601369
val score: 0.47104551610474343
----------------------------------------
XGBoost
train score: 0.8648788545746917
val score: 0.5247891447491386
----------------------------------------


In [130]:
# loggg
# 1) Random Forest         0.620
# 2) Gradient Boosting     0.615
# 3) XGBoost               0.601
# KNN                      0.507

In [131]:
'''from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import RandomizedSearchCV


xgb_pipeline = Pipeline([
    ('preprocessing', preprocessing),
    ('model', XGBRegressor(random_state=42, n_jobs=-1, verbosity=0)),
])
 
xgb_model = TransformedTargetRegressor(
    regressor=xgb_pipeline,
    func=np.log1p,
    inverse_func=np.expm1,
)

param_grid = {
    'regressor__model__max_depth':        [4, 5, 6],
    'regressor__model__learning_rate':    [0.03, 0.05, 0.1],
    'regressor__model__n_estimators':     [300, 500, 700],
    'regressor__model__subsample':        [0.7, 0.8, 0.9],
    'regressor__model__colsample_bytree': [0.7, 0.8, 0.9],
    'regressor__model__min_child_weight': [1, 3, 5],
    'regressor__model__reg_lambda':       [1, 5, 10],
}
 
cv = KFold(n_splits=5, shuffle=True, random_state=42)
 
search = RandomizedSearchCV(
    estimator=xgb_model,
    param_distributions=param_grid,
    n_iter=40,
    cv=cv,
    scoring='r2',
    random_state=42,
    n_jobs=-1,
    verbose=1,
)
 
 
search.fit(x_train, y_train)

print("Best CV score:", search.best_score_)
print("Best params:", search.best_params_)'''


'from sklearn.neighbors import KNeighborsRegressor\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.compose import TransformedTargetRegressor\nfrom sklearn.model_selection import RandomizedSearchCV\n\n\nxgb_pipeline = Pipeline([\n    (\'preprocessing\', preprocessing),\n    (\'model\', XGBRegressor(random_state=42, n_jobs=-1, verbosity=0)),\n])\n\nxgb_model = TransformedTargetRegressor(\n    regressor=xgb_pipeline,\n    func=np.log1p,\n    inverse_func=np.expm1,\n)\n\nparam_grid = {\n    \'regressor__model__max_depth\':        [4, 5, 6],\n    \'regressor__model__learning_rate\':    [0.03, 0.05, 0.1],\n    \'regressor__model__n_estimators\':     [300, 500, 700],\n    \'regressor__model__subsample\':        [0.7, 0.8, 0.9],\n    \'regressor__model__colsample_bytree\': [0.7, 0.8, 0.9],\n    \'regressor__model__min_child_weight\': [1, 3, 5],\n    \'regressor__model__reg_lambda\':       [1, 5, 10],\n}\n\ncv = KFold(n_splits=5, shuffle=True, random_state=42)\n\nsearch = RandomizedSearchC

In [132]:
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import RandomizedSearchCV


rf_pipeline = Pipeline(steps=[
    ('preprocessing', preprocessing),
    ('model', RandomForestRegressor(random_state=42, n_jobs=-1))
])

model = TransformedTargetRegressor(
    regressor=rf_pipeline,
    func=np.log1p,
    inverse_func=np.expm1
)

param_dist = {
    'regressor__model__n_estimators': [200, 400, 600, 800, 1000],
    'regressor__model__max_depth': [None, 5, 10, 15, 20, 30],
    'regressor__model__min_samples_split': [2, 5, 10, 20],
    'regressor__model__min_samples_leaf': [1, 2, 4, 8],
    'regressor__model__max_features': ['sqrt', 'log2', 0.5, 0.8],
    'regressor__model__bootstrap': [True, False]
}


search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_dist,
    n_iter=40,
    cv=5,
    scoring='r2',
    verbose=2,
    random_state=42,
    n_jobs=-1,
    return_train_score=True
)

search.fit(x, y)


print("Best R2:", search.best_score_)
print("Best Params:", search.best_params_)

Fitting 5 folds for each of 40 candidates, totalling 200 fits
Best R2: 0.5002049599543961
Best Params: {'regressor__model__n_estimators': 400, 'regressor__model__min_samples_split': 2, 'regressor__model__min_samples_leaf': 1, 'regressor__model__max_features': 0.8, 'regressor__model__max_depth': 10, 'regressor__model__bootstrap': True}


In [133]:
'''
import numpy as np
from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import RandomizedSearchCV

xgb_pipeline = Pipeline(steps=[
    ('preprocessing', preprocessing),
    ('model', XGBRegressor(
        objective='reg:squarederror',
        random_state=42,
        n_jobs=-1
    ))
])

model = TransformedTargetRegressor(
    regressor=xgb_pipeline,
    func=np.log1p,
    inverse_func=np.expm1
)

param_dist = {
    'regressor__model__n_estimators': [200, 400, 600, 800, 1000],
    'regressor__model__max_depth': [3, 5, 7, 9],
    'regressor__model__learning_rate': [0.01, 0.03, 0.05, 0.1],
    'regressor__model__subsample': [0.6, 0.8, 1.0],
    'regressor__model__colsample_bytree': [0.6, 0.8, 1.0],
    'regressor__model__min_child_weight': [1, 3, 5, 7],
    'regressor__model__gamma': [0, 0.1, 0.3, 0.5]
}

search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_dist,
    n_iter=40,        
    cv=5,
    scoring='r2',
    verbose=2,
    random_state=42,
    n_jobs=-1,
    return_train_score=True
)

search.fit(x, y)

print("Best R2:", search.best_score_)
print("Best Params:", search.best_params_)

Best R2: 0.4767723997157445
Best Params: {'regressor__model__subsample': 0.6, 'regressor__model__n_estimators': 400, 'regressor__model__min_child_weight': 5, 'regressor__model__max_depth': 3, 'regressor__model__learning_rate': 0.05, 'regressor__model__gamma': 0.3, 'regressor__model__colsample_bytree': 0.6}'''


'\nimport numpy as np\nfrom xgboost import XGBRegressor\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.compose import TransformedTargetRegressor\nfrom sklearn.model_selection import RandomizedSearchCV\n\nxgb_pipeline = Pipeline(steps=[\n    (\'preprocessing\', preprocessing),\n    (\'model\', XGBRegressor(\n        objective=\'reg:squarederror\',\n        random_state=42,\n        n_jobs=-1\n    ))\n])\n\nmodel = TransformedTargetRegressor(\n    regressor=xgb_pipeline,\n    func=np.log1p,\n    inverse_func=np.expm1\n)\n\nparam_dist = {\n    \'regressor__model__n_estimators\': [200, 400, 600, 800, 1000],\n    \'regressor__model__max_depth\': [3, 5, 7, 9],\n    \'regressor__model__learning_rate\': [0.01, 0.03, 0.05, 0.1],\n    \'regressor__model__subsample\': [0.6, 0.8, 1.0],\n    \'regressor__model__colsample_bytree\': [0.6, 0.8, 1.0],\n    \'regressor__model__min_child_weight\': [1, 3, 5, 7],\n    \'regressor__model__gamma\': [0, 0.1, 0.3, 0.5]\n}\n\nsearch = RandomizedSearchCV(\

In [134]:
final_model = search.best_estimator_
final_model.fit(x,y)
final_model

,"regressor regressor: object, default=NoneRegressor object such as derived from:class:`~sklearn.base.RegressorMixin`. This regressor willautomatically be cloned each time prior to fitting. If `regressor isNone`, :class:`~sklearn.linear_model.LinearRegression` is created and used.",Pipeline(step...m_state=42))])
,"transformer transformer: object, default=NoneEstimator object such as derived from:class:`~sklearn.base.TransformerMixin`. Cannot be set at the same timeas `func` and `inverse_func`. If `transformer is None` as well as`func` and `inverse_func`, the transformer will be an identitytransformer. Note that the transformer will be cloned during fitting.Also, the transformer is restricting `y` to be a numpy array.",None
,"func func: function, default=NoneFunction to apply to `y` before passing to :meth:`fit`. Cannot be setat the same time as `transformer`. If `func is None`, the function used will bethe identity function. If `func` is set, `inverse_func` also needs to beprovided. The function needs to return a 2-dimensional array.",<ufunc 'log1p'>
,"inverse_func inverse_func: function, default=NoneFunction to apply to the prediction of the regressor. Cannot be set atthe same time as `transformer`. The inverse function is used to returnpredictions to the same space of the original training labels. If`inverse_func` is set, `func` also needs to be provided. The inversefunction needs to return a 2-dimensional array.",<ufunc 'expm1'>
,"check_inverse check_inverse: bool, default=TrueWhether to check that `transform` followed by `inverse_transform`or `func` followed by `inverse_func` leads to the original targets.",True
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('ohe', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transform

In [135]:
preds = final_model.predict(x.head(5))

pd.DataFrame({'actual': y.head(5).values, 'predicted': preds.round(0)})

,actual,predicted
0,239900.0,208134.0
1,259900.0,244995.0
2,342500.0,231141.0
3,335000.0,291736.0
4,250000.0,232647.0


In [136]:
print(((df.isnull().sum()/len(df))*100).sort_values(ascending=False))

df['rentestimate'] = df['rentestimate'].fillna(df['rentestimate'].median())
df['lotarea']       = df['lotarea'].fillna(df['lotarea'].median())
df['lot_to_area_ratio'] = df['lotarea'] / df['area']  
df['bathroom']       = df['bathroom'].fillna(df['bathroom'].median())
df['bedroom']        = df['bedroom'].fillna(df['bedroom'].median())
df.to_csv('clean_price_prediction_df.csv', index=False)

rentestimate         26.343896
lotarea               4.916586
lot_to_area_ratio     4.916586
bathroom              0.485480
bedroom               0.061788
city                  0.000000
state                 0.000000
area                  0.000000
latitude              0.000000
longitude             0.000000
listedprice           0.000000
dtype: float64


In [137]:
print(((df.isnull().sum()/len(df))*100).sort_values(ascending=False))

state                0.0
city                 0.0
bedroom              0.0
bathroom             0.0
area                 0.0
lotarea              0.0
rentestimate         0.0
latitude             0.0
longitude            0.0
listedprice          0.0
lot_to_area_ratio    0.0
dtype: float64


In [138]:
import joblib 

joblib.dump(final_model,'house_price_model.pkl')

['house_price_model.pkl']

In [5]:
%%writefile house_price_app.py
import streamlit as st
import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import joblib
import gzip
import os
import numpy as np

st.set_page_config(page_title=" House Price Analytics", layout="wide")

st.sidebar.title(" House Price Dashboard")

@st.cache_data
def load_data():
    df = pd.read_csv('clean_price_prediction_df.csv')  
    return df

df = load_data()

df['price_per_sqft'] = df['listedprice'] / df['area']
df['lot_to_area_ratio'] = df['lotarea'] / df['area']
df['bed_bath_ratio'] = df['bedroom'] / df['bathroom'].replace(0, np.nan)

page = st.sidebar.radio(
    'Pages',
    ['Main Dashboard', 'Price Factors Analysis', "Prediction"]
)

st.sidebar.title('Filters')

state_filter = st.sidebar.multiselect(
    'State', options=df['state'].unique(), default=df['state'].unique()
)

city_filter = st.sidebar.multiselect(
    'City', options=df['city'].unique(), default=df['city'].unique()
)

filtered_df = df[
    (df['state'].isin(state_filter)) &
    (df['city'].isin(city_filter))
]

if page == "Main Dashboard":
    st.title(" House Price Main Dashboard")
    st.markdown("---")

    col1, col2, col3 = st.columns(3)
    col1.metric(" Avg Listed Price", f"${filtered_df['listedprice'].mean():,.0f}")
    col2.metric(" Avg Area (sqft)", f"{filtered_df['area'].mean():,.0f}")
    col3.metric(" Avg Price/sqft", f"${filtered_df['price_per_sqft'].mean():,.0f}")

    st.subheader("Dataset Preview")
    st.dataframe(filtered_df.head(10))

elif page == "Price Factors Analysis":
    st.title(" Price Factors Analysis")
    tab1, tab2, tab3 = st.tabs(["Size & Rooms", "Location & Market", "Advanced Insights"])

    with tab1:
        st.subheader("Impact of Size & Rooms")
        col1, col2 = st.columns(2)

        with col1:
            fig = px.scatter(filtered_df, x='area', y='listedprice', color='bedroom',opacity=0.6,
                           title='Area vs Price (colored by Bedrooms)',
                           trendline='ols',color_continuous_scale='Viridis')
            st.plotly_chart(fig, use_container_width=True)

        with col2:
            bed_avg = filtered_df.groupby('bedroom')['listedprice'].mean().reset_index()
            fig = px.bar(bed_avg, x='bedroom', y='listedprice',title='Avg Price by Number of Bedrooms',
                color='listedprice',color_continuous_scale='Blues'
            )
        fig.update_layout(coloraxis_showscale=False)
        st.plotly_chart(fig, use_container_width=True)

        
        bath_avg = filtered_df.groupby('bathroom')['listedprice'].mean().reset_index()
        fig = px.bar(bath_avg, x='bathroom', y='listedprice', 
                    title='Avg Price by Bathrooms', color='listedprice',color_continuous_scale='YlGnBu')
        fig.update_traces(width=0.8,textposition='outside')
        fig.update_layout(coloraxis_showscale=False,bargap=0.05,template='plotly_white')
        st.plotly_chart(fig, use_container_width=True)

    with tab2:
        st.subheader("Location & Market Influence")
        col1, col2 = st.columns(2)

        with col1:
            state_price = filtered_df.groupby('state')['listedprice'].mean().sort_values(ascending=False).head(15)
            fig = px.bar(x=state_price.index, y=state_price.values, 
                        title='Avg Price by State (Top 15)', color_discrete_sequence=px.colors.sequential.Plasma,
                labels={'x': 'State', 'y': 'Avg Listed Price'},)
            st.plotly_chart(fig, use_container_width=True)

        with col2:
            fig = px.scatter(
            filtered_df.sample(min(1000, len(filtered_df))),
            x='longitude',y='latitude',color='listedprice',
            size='area',color_continuous_scale=px.colors.sequential.Mint,title='Geographic Distribution of Prices',)
            st.plotly_chart(fig, use_container_width=True)
        
        filtered_df['log_price'] = np.log1p(filtered_df['listedprice'])

        fig = px.histogram(filtered_df,x='log_price',nbins=50,marginal='box',
            title='Log Price Distribution',
            color_discrete_sequence=['#C62828']
        )

        fig.update_layout(title='House Price Distribution',bargap=0.05)
        st.plotly_chart(fig, use_container_width=True)

    with tab3:
        st.subheader("Correlations & Other Factors")
        num_cols = ['bedroom', 'bathroom', 'area', 'lotarea', 'rentestimate', 'price_per_sqft', 'listedprice']
        import plotly.express as px

        corr_price = (
            filtered_df.select_dtypes(include='number')
            .corr()[['listedprice']]
            .sort_values('listedprice', ascending=False)
        )

        fig = px.imshow(corr_price,text_auto='.2f',color_continuous_scale='Reds',
            aspect='auto'
        )

        fig.update_layout(
            title='Correlation with House Price',
            height=600
        )
        st.plotly_chart(fig, use_container_width=True)

        fig = px.scatter(filtered_df, x='rentestimate', y='listedprice', 
                        title='Rent Estimate vs Listed Price', trendline='ols')
        st.plotly_chart(fig, use_container_width=True)
    
elif page == "Prediction":
    st.title('House Price Prediction')

    df = pd.read_csv('clean_price_prediction_df.csv')
    #model = joblib.load('house_price_model.pkl')


    @st.cache_resource
    def load_model():
        model_path = "house_price_model.pkl.gz"
        
        if not os.path.exists(model_path):
            st.error(" Model file not found!")
            st.stop()
        
        try:
            with gzip.open(model_path, 'rb') as f:
                model = joblib.load(f)
            return model
        except Exception as e:
            st.error(f" Error loading model: {str(e)}")
            st.stop()

    
    model = load_model()

    st.dataframe(df.head())

    col1, col2 = st.columns(2)
    with col1:
        state    = st.selectbox('State', sorted(df['state'].dropna().unique()))
        cities   = sorted(df.loc[df['state']==state,'city'].dropna().unique())
        city     = st.selectbox('City', cities)
        bedroom  = st.number_input('Bedrooms', 1, 10, 3)
        bathroom = st.number_input('Bathrooms', 1.0, 10.0, 2.0, step=0.5)
        area     = st.number_input('Area (sq ft)', 200.0, 20000.0, 1800.0)
    with col2:
        lotarea       = st.number_input('Lot Area (acres)', 0.0, 50.0, 0.3)
        rentestimate  = st.number_input('Rent Estimate', 0.0, 20000.0, 1800.0)
        latitude      = st.number_input('Latitude',  -90.0, 90.0,  float(df['latitude'].median()))
        longitude     = st.number_input('Longitude', -180.0, 180.0, float(df['longitude'].median()))

    if st.button('Predict price'):
        lot_ratio = lotarea / area if area else None
        new_row = pd.DataFrame([{
            'state': state, 'city': city,
            'bedroom': bedroom, 'bathroom': bathroom,
            'area': area, 'lotarea': lotarea,
            'rentestimate': rentestimate,
            'latitude': latitude, 'longitude': longitude,
            'lot_to_area_ratio': lot_ratio,
        }])
        pred = model.predict(new_row)[0]
        st.success(f'Predicted listed price: ${pred:,.0f}')

Overwriting house_price_app.py


In [3]:
! streamlit run house_price_app.py

^C


In [2]:
! pip list

Package                   Version
------------------------- -----------
altair                    6.0.0
anyio                     4.12.1
argon2-cffi               25.1.0
argon2-cffi-bindings      25.1.0
arrow                     1.4.0
asttokens                 3.0.1
async-lru                 2.1.0
attrs                     25.4.0
babel                     2.18.0
beautifulsoup4            4.14.3
bleach                    6.3.0
blinker                   1.9.0
cachetools                6.2.6
catboost                  1.2.10
category_encoders         2.9.0
certifi                   2026.1.4
cffi                      2.0.0
charset-normalizer        3.4.4
click                     8.3.1
colorama                  0.4.6
comm                      0.2.3
contourpy                 1.3.3
cycler                    0.12.1
datasist                  1.5.3
debugpy                   1.8.19
decorator                 5.2.1
defusedxml                0.7.1
et_xmlfile                2.0.0
executing           

In [3]:
! pip freeze > requirements.txt

In [4]:
import joblib
import gzip
import shutil
import os


model_path = 'house_price_model.pkl'

with open(model_path, 'rb') as f_in:
    with gzip.open('house_price_model.pkl.gz', 'wb') as f_out:
        shutil.copyfileobj(f_in, f_out)

size_before = os.path.getsize(model_path) / (1024*1024)
size_after = os.path.getsize('house_price_model.pkl.gz') / (1024*1024)

print(f"الحجم قبل الضغط: {size_before:.2f} MB")
print(f"الحجم بعد الضغط: {size_after:.2f} MB")

الحجم قبل الضغط: 33.74 MB
الحجم بعد الضغط: 10.36 MB
